## Run this notebook with seacells
```pip install squidpy shapely kneed opencv-python```

## Integration of ST and histopathological data

In [ ]:
import cv2

from kneed import KneeLocator

import json

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.patches as patches
from matplotlib.patches import Polygon, Wedge, RegularPolygon, Rectangle

import networkx as nx
import numpy as np

import os

import pandas as pd
from pathlib import Path
from PIL import Image

import scanpy as sc
import scipy.sparse as sparse
from scipy.stats import t
import seaborn as sns
from shapely.ops import unary_union
from shapely.geometry import Point, MultiPolygon, Polygon as ShapelyPolygon
from skimage.filters import threshold_otsu
import squidpy as sq

# Helper functions

def compute_thresholds(scores):
    nz = scores[scores > 0]
    q75 = np.quantile(nz, .75) if len(nz) > 0 else 0
    otsu = float(threshold_otsu(nz)) if len(nz) > 0 else q75
    sk = np.sort(nz)
    kn = KneeLocator(np.arange(len(sk)), sk, S=1.0, curve="convex", direction="increasing")
    knee = float(sk[kn.knee]) if kn.knee is not None else q75
    return {"q75": q75, "otsu": otsu, "knee": knee}

def build_method_polygons(regs_by_method, coords):
    out = {}
    for m, comps in regs_by_method.items():
        polys = []
        for comp in comps:
            circs = [Point(coords.loc[i, "y_img"], coords.loc[i, "x_img"]).buffer(SPOT_RADIUS) for i in comp]
            polys.append(unary_union(circs).simplify(SPOT_RADIUS / 2))
        out[m] = polys
    return out

def annotate_spot_types(adata, spots):
    def expr(spot, g):
        return adata[spot, g].X[0, 0] if g in adata.var_names else 0.0

    annotations = []
    for spot in spots:
        spot_annotation = {
            "spot": spot,
            "beta": (expr(spot, "Ins1") > 0) or (expr(spot, "Ins2") > 0),
            "alpha": expr(spot, "Gcg") > 0,
            "gamma": expr(spot, "Sst") > 0,
            "panleuko": expr(spot, "Ptprc") > 0,
        }

        cd3_any = any(expr(spot, g) > 0 for g in ("Cd3", "Cd3d", "Cd3e", "Cd3g"))
        has_cd8 = (expr(spot, "Cd8a") > 0) or (expr(spot, "Cd8b1") > 0)
        has_cd4 = expr(spot, "Cd4") > 0

        spot_annotation.update({
            "Tcell": spot_annotation["panleuko"] and cd3_any,
            "CD8pos": spot_annotation["panleuko"] and cd3_any and has_cd8 and not has_cd4,
            "cytoCD8": spot_annotation["panleuko"] and cd3_any and has_cd8 and not has_cd4 and expr(spot, "Gzma") > 0 and expr(spot, "Gzmb") > 0,
            "GLP1Rpos": expr(spot, "Glp1r") > 0,
        })

        annotations.append(spot_annotation)

    return pd.DataFrame(annotations)

def extract_blob_border_pixels(tiff_path):
    """Extract border pixels of blobs from a TIFF image."""
    img = Image.open(tiff_path).convert('RGB')
    img_np = np.array(img)

    pixels, counts = np.unique(img_np.reshape(-1, 3), axis=0, return_counts=True)
    background_color = pixels[np.argmax(counts)]

    mask = np.any(img_np != background_color, axis=-1).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    return [cnt.reshape(-1, 2).tolist() for cnt in contours]


def scaled_borders(sample, label, tiff_dir, png_shape):
    """Scale TIFF blob borders to PNG coordinates."""
    tiff_path = os.path.join(tiff_dir, f"{sample}_tissue_hires_image_{label}.ome.tiff")
    borders = extract_blob_border_pixels(tiff_path)

    with Image.open(tiff_path) as im:
        tiff_w, tiff_h = im.size

    sx, sy = png_shape[1] / tiff_w, png_shape[0] / tiff_h

    return [[(int(x * sx), int(y * sy)) for x, y in border] for border in borders]


def create_mask_from_borders(borders, shape):
    mask = np.zeros(shape, dtype=np.uint8)
    for border in borders:
        cv2.drawContours(mask, [np.array(border)], -1, color=255, thickness=-1)
    return mask


def check_overlaps(borders_dict, img_shape):
    masks = {label: create_mask_from_borders(borders, img_shape)
             for label, borders in borders_dict.items()}

    overlaps = {}
    labels = list(masks.keys())

    for i, label1 in enumerate(labels):
        for label2 in labels[i + 1:]:
            overlap_area = cv2.bitwise_and(masks[label1], masks[label2])
            if np.any(overlap_area):
                overlaps.setdefault(label1, []).append(label2)

    return overlaps

def convert_to_dict_format(laure_polygons_with_ids):
    laure_polygons_dict = []
    for label, poly in laure_polygons_with_ids:
        coords = list(poly.exterior.coords)
        laure_polygons_dict.append({
            'id': label,
            'polygon': coords
        })
    return laure_polygons_dict

## Visualize overlap

In [ ]:
# SETTINGS
INPUT_DIR = Path("results/intermediate/GraphST_legacy/")
IMG_DIR = Path("data/cytassistimages/hires/")
OUTDIR = Path("results/figures/")

# --- SETTINGS ---------------------------------------------------
COND_MAP = {
    "W12F":"control_12W","Mouse1":"control_12W","Mouse2":"control_12W",
    "FFPE315":"control_17W","FFPE319":"control_17W","FFPE320":"control_17W",
    "FFPE317":"aCD3_17W","FFPE318":"aCD3_17W","FFPE322":"aCD3_17W",
    "FFPE801":"E2GLP1_17W","FFPE802":"E2GLP1_17W","FFPE803":"E2GLP1_17W",
    "FFPE316":"combo_17W","FFPE321":"combo_17W","FFPE323":"combo_17W"
}
EXT_MARKERS = {
    "islet": [
        ["Ins1","Ins2","Gcg","Sst","Chga"], 
        ["Pdx1","Slc2a2","Glut2"],
        ["Ucn3","Mafa","Pax6","Foxo1","Ngn3","Nkx6-1"]
    ],
    "immune": [
        ["Ptprc","Cd45"], ["Cd8","Cd8a","Cd8b1","Cd4","Cd3","Ccl6","Gzma","Gzmb"],
        ["Pdcd1","Lag3","Havcr2","Tigit","Ctla4"],
        ["Atxn1","Casp3","Ccr7","Cd14","Cd163","Cd19","Cd1d1","Cd247","Cd34",
         "Cd3d","Cd3e","Cd3g","Cd69","Cd9","Eng","Fcer2a","Fcgr4","Hck",
         "Hmox1","Hmox2","Il15","Il7r","Igax","Itgb2","Kit","Klf6","Ncam1",
         "Nrp1","Pdgfrb","Pecam1","Ppp1r9b","Prprc","Sell","Slamf1","Slc9a9",
         "Sod2","Srxn1","Tcirg1","Thy1","Txnrd1"]
    ]
}
methods = ["q75","otsu","knee"]
colors = {"q75":"red","otsu":"green","knee":"purple"}
MIN_REGION_SIZE = 5
spot_radius = 5
margin = spot_radius*3
margin_Laure = 20

sanity_genes = ["Ins1","Ins2","Gcg","Sst","Chga"]

metadata_drill = 'otsu'

INLINE_PLOTS = False

spot_radius = 5
MIN_REGION_SIZE = 5
methods = ['q75', 'otsu', 'knee']
colors = {'q75': 'red', 'otsu': 'green', 'knee': 'purple', 'Laure': 'magenta'}

all_cords = []

labels = [
    "Questionable islets", "Score 0 islet", "Score 1 islet",
    "Score 2 islet", "Score 3 islet",
]

sample_ids = [
    "FFPE315", "FFPE316", "FFPE317", "FFPE318", "FFPE319", "FFPE320",
    "FFPE322", "FFPE323", "FFPE801", "FFPE802", "FFPE803", "Mouse1",
    "Mouse2", "W12F",
]

tiff_dir = "data/Annotations_complete/"
png_dir = "data/cytassistimages/hires/"


In [ ]:
scaled_laure_polygons = {}

for sample in sample_ids:
    png_path = os.path.join(png_dir, f"tissue_hires_image{sample}.png")
    png_img = mpimg.imread(png_path)
    png_shape = png_img.shape[:2]

    all_scaled_borders = {
        label: scaled_borders(sample, label, tiff_dir, png_shape)
        for label in labels
    }

    # overlap_dict = check_overlaps(all_scaled_borders, png_shape)

    # print(f"Overlaps detected in {sample}:")
    # if overlap_dict:
    #     for key, vals in overlap_dict.items():
    #         print(f" - {key} overlaps with {vals}")
    # else:
    #     print(" - No overlaps found.")

    # Laure’s manual annotations with unified region IDs
    region_counter = 0
    laure_polygons_with_ids = []
    for key in ['Questionable islets', 'Score 0 islet', 'Score 1 islet', 'Score 2 islet', 'Score 3 islet']:
        prefix = key.split()[1] if key.startswith('Score') else 'Q'
        polygons = all_scaled_borders.get(key, [])
        for border in polygons:
            label = f"Laure_{prefix}_{region_counter}"
            laure_polygons_with_ids.append((label, ShapelyPolygon(border)))
            region_counter += 1
    
    scaled_laure_polygons[sample] = convert_to_dict_format(laure_polygons_with_ids)

# Process files
for fn in sorted(INPUT_DIR.glob("*k12.h5ad")):
    sample = fn.stem
    st = sample.split('_')[0]
    sample_dir = OUTDIR / sample
    sample_dir.mkdir(parents=True, exist_ok=True)

    ad = sc.read_h5ad(fn)

    # 1) compute panel-max & record raw expr
    for key, panels in EXT_MARKERS.items():
        sc_col = f"{key}_score"
        ad.obs[sc_col] = 0.0
        for panel in panels:
            genes = [g for g in panel if g in ad.var_names]
            if not genes: continue
            Xp = ad[:,genes].X
            vals = Xp.toarray().max(1) if sparse.issparse(Xp) else Xp.max(1)
            ad.obs[sc_col] = np.nan_to_num(vals.ravel())
            for g in genes:
                Xg = ad[:,g].X
                ad.obs[f"{g}_expr"] = Xg.toarray().ravel()
            break

    # 2) thresholds & histograms
    thr = {k:{} for k in EXT_MARKERS}
    for key in EXT_MARKERS:
        scores = ad.obs[f"{key}_score"].values
        nz     = scores[scores>0]
        q75    = np.quantile(scores, 0.75)
        if q75 <= 0 and len(nz)>0: q75 = nz.min()
        thr[key]["q75"] = q75
        thr[key]["otsu"] = float(threshold_otsu(nz)) if len(nz)>0 else q75
        sk = np.sort(nz)
        k  = KneeLocator(np.arange(len(sk)), sk, S=1.0, curve="convex", direction="increasing")
        thr[key]["knee"] = float(sk[k.knee]) if k.knee is not None else q75

    # 3) spot coords → image space (direct scaling)
    coords = pd.DataFrame({
        "spot":  ad.obs_names,
        "pos_x": ad.obs["pos_x"].values,
        "pos_y": ad.obs["pos_y"].values
    })
    coords["x_img"] = 2000 * coords.pos_x / 600
    coords["y_img"] = 2000 * coords.pos_y / 600

    # load hires image
    img = mpimg.imread(f"data/cytassistimages/hires/tissue_hires_image{st}.png")

    # 4) scatter all spots per method - commented out for speed
    for m in methods:
        fig, ax = plt.subplots(figsize=(8,8))
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"{sample} — spots ({m})")
        # all spots
        ax.scatter(coords.y_img, coords.x_img,
                   s=spot_radius, c="lightblue", alpha=0.6,
                   edgecolors="none")
        islet_pos  = ad.obs["islet_score"].values > thr["islet"][m]
        immune_pos = ad.obs["immune_score"].values > thr["immune"][m]
        ax.scatter(coords.y_img[islet_pos], coords.x_img[islet_pos],
                   s=spot_radius, c="blue", label="islet", edgecolors="none")
        ax.scatter(coords.y_img[immune_pos], coords.x_img[immune_pos],
                   s=spot_radius, c="red",  label="immune", edgecolors="none")
        ax.legend(loc="upper right", title=m)
        if INLINE_PLOTS:
            plt.show()
        else:
            (sample_dir / "spots_all").mkdir(parents=True, exist_ok=True)
            fig.savefig(sample_dir / "spots_all" / f"{sample}_spots_{m}.png", dpi=150)
            fig.savefig(sample_dir / "spots_all" / f"{sample}_spots_{m}.svg")
            plt.close(fig)

    # 5) compute islet regions per method
    if "spatial_connectivities" not in ad.obsp:
        sq.gr.spatial_neighbors(ad, coord_type="grid", n_rings=1)
    A = ad.obsp['spatial_connectivities'].tocoo()
    neighbors = {i: set(A.getrow(i).indices) for i in range(ad.n_obs)}
    regs_by_method = {}
    for m in methods:
        flag = ad.obs["islet_score"].values > thr["islet"][m]
        iso  = np.where(flag)[0]
        G = nx.Graph(); G.add_nodes_from(iso)
        for i in iso:
            for j in neighbors[i]:
                if j in iso: G.add_edge(i, j)
        regs_by_method[m] = [sorted(c) for c in nx.connected_components(G)
                             if len(c)>=MIN_REGION_SIZE]

    coords = pd.DataFrame({
        'spot': ad.obs_names,
        'x_img': 2000 * ad.obs["pos_x"].values / 600,
        'y_img': 2000 * ad.obs["pos_y"].values / 600,
        'islet_score': ad.obs['islet_score']
    })

    try:
        # Laure assignment
        laure_polygons_shapely = {r['id']: ShapelyPolygon(r['polygon']).buffer(margin_Laure) for r in scaled_laure_polygons[st]}
        coords['Laure_region_id'] = None
        for idx, spot in coords.iterrows():
            point = Point(spot.x_img, spot.y_img)
            for region_id, poly in laure_polygons_shapely.items():
                if poly.contains(point):
                    coords.loc[idx,'Laure_region_id'] = region_id
    
        for m in methods:
            coords[f'region_{m}'] = None
            for region_id, comp in enumerate(regs_by_method[m]):
                coords.loc[coords.index[np.concatenate(regs_by_method[m])], f'region_{m}'] = region_id
    
        all_cords.append(coords)
    
        laure_scores = coords.dropna(subset=['Laure_region_id'])['islet_score']
        thr['islet']['Laure'] = laure_scores.min() if not laure_scores.empty else np.nan
    
        # Plot with Laure threshold and spots satisfying it
        fig, ax = plt.subplots(figsize=(8,8))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"{sample} — Laure Threshold Spots")
        
        # All spots
        ax.scatter(coords.y_img, coords.x_img,
                   s=spot_radius, c="lightblue", alpha=0.6,
                   edgecolors="none")
        
        # Spots satisfying Laure threshold
        laure_threshold = thr['islet']['Laure']
        laure_spots = ad.obs['islet_score'].values >= laure_threshold
        
        ax.scatter(coords.y_img[laure_spots], coords.x_img[laure_spots],
                   s=spot_radius, c="magenta", label="Laure threshold", edgecolors="none")
        
        # Laure regions overlay
        for region in scaled_laure_polygons[st]:
            poly_patch = patches.Polygon(region['polygon'], fill=False, edgecolor='magenta', linestyle='--', lw=2)
            ax.add_patch(poly_patch)
        
        ax.legend(loc="upper right")
        if INLINE_PLOTS:
            plt.show()
        else:
            plt.close()
    
        # Plot thresholds
        plt.figure(figsize=(8, 5))
        plt.hist(scores, bins=50, alpha=0.6, color='gray', label='All spots',log=True)
        for method, val in thr['islet'].items():
            plt.axvline(val, linestyle='--', color=colors[method], label=f'{method}={val:.2f}')
        plt.xlabel('Islet score')
        plt.ylabel('Count')
        plt.legend()
        plt.title(f'{sample} - Thresholds Comparison')
        plt.tight_layout()
        if INLINE_PLOTS:
            plt.show()
        else:
            plt.savefig(sample_dir / f"{sample}_threshold_comparison.png")
            plt.close()
    
        # Overlay manual and automatic regions
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(img)
    
        for region in scaled_laure_polygons[st]:
            poly_patch = patches.Polygon(region['polygon'], fill=False, edgecolor='magenta', linestyle='--', lw=2)
            ax.add_patch(poly_patch)
    
        sq.gr.spatial_neighbors(ad, coord_type='grid', n_rings=1)
        A = ad.obsp['spatial_connectivities'].tocoo()
        neighbors = {i: set(A.getrow(i).indices) for i in range(ad.n_obs)}
    
        for method in methods:
            flag = scores > thr['islet'][method]
            iso = np.where(flag)[0]
            G = nx.Graph(); G.add_nodes_from(iso)
            for i in iso:
                for j in neighbors[i]:
                    if j in iso:
                        G.add_edge(i, j)
            regions = [c for c in nx.connected_components(G) if len(c) >= MIN_REGION_SIZE]
    
            for region in regions:
                spots = coords.iloc[list(region)]
                circles = [Point(y, x).buffer(spot_radius) for x, y in zip(spots.x_img, spots.y_img)]
                unified = unary_union(circles).simplify(spot_radius / 2)
                polys = unified.geoms if isinstance(unified, MultiPolygon) else [unified]
    
                for poly in polys:
                    xs, ys = poly.exterior.xy
                    ax.add_patch(patches.Polygon(list(zip(xs, ys)), fill=False, edgecolor=colors[method], lw=2))
    
        ax.set_title(f'{sample} - Regions Overlay')
        plt.tight_layout()
        if INLINE_PLOTS:
            plt.show()
        else:
            plt.savefig(sample_dir / f"{sample}_regions_overlay.png")
            plt.close(fig) 
    except:
        continue
    

## Create pseudobulks metadata and count files

In [ ]:
INPUT_DIR = Path("results/intermediate/GraphST_legacy/")
EXT_MARKERS = {"islet": [
    ["Ins1", "Ins2", "Gcg", "Sst", "Chga"],
    ["Pdx1", "Slc2a2", "Glut2"],
    ["Ucn3", "Mafa", "Pax6", "Foxo1", "Ngn3", "Nkx6-1"]
]}

METHOD = "q75"
SPOT_RADIUS = 5
MIN_REGION_SIZE = 5
margin_Laure = 20

# Load Laure annotations
with open("objects/samples_laure_borders.json") as f:
    samples = json.load(f)

# Process samples
for sample_id, laure_regions in samples.items():
    print(f"\nProcessing sample {sample_id}")

    ad = sc.read_h5ad(INPUT_DIR / f"{sample_id}_CARDobj_step1_k12.h5ad")
    ad.obs["islet_score"] = 0.0
    for panel in EXT_MARKERS["islet"]:
        genes = [g for g in panel if g in ad.var_names]
        if not genes:
            continue
        ad.obs["islet_score"] = np.nan_to_num(ad[:, genes].X.toarray().max(1).ravel())
        break

    thr = compute_thresholds(ad.obs["islet_score"].values)
    if "spatial_connectivities" not in ad.obsp:
        sq.gr.spatial_neighbors(ad, coord_type="grid", n_rings=1)

    A = ad.obsp["spatial_connectivities"].tocoo()
    neigh = {i: set(A.getrow(i).indices) for i in range(ad.n_obs)}

    iso = np.where(ad.obs["islet_score"] > thr[METHOD])[0]
    G = nx.Graph()
    G.add_nodes_from(iso)
    for i in iso:
        for j in neigh[i]:
            if j in iso:
                G.add_edge(i, j)
    regs_by_method = [c for c in nx.connected_components(G) if len(c) >= MIN_REGION_SIZE]

    xs = 2000 * ad.obs["pos_x"].values / 600
    ys = 2000 * ad.obs["pos_y"].values / 600
    spots = np.array(ad.obs_names)

    region_polys = build_method_polygons({METHOD: regs_by_method}, pd.DataFrame({"x_img": xs, "y_img": ys}))[METHOD]

    laure_polys_dict = {
        region_id: ShapelyPolygon(pts).buffer(margin_Laure)
        for region_id, pts in laure_regions.items() if len(pts) >= 3
    }

    spot_region_ids = np.full(ad.n_obs, fill_value=-1)
    spot_laure_ids = np.array(["none"] * ad.n_obs, dtype=object)

    for rid, comp in enumerate(regs_by_method):
        region_poly = region_polys[rid]
        overlapping_laure = [lname for lname, lpoly in laure_polys_dict.items() if region_poly.intersects(lpoly)]
        laure_id = overlapping_laure[0] if overlapping_laure else "none"
        for idx in comp:
            spot_region_ids[idx] = rid
            spot_laure_ids[idx] = laure_id

    mask_m = ad.obs["islet_score"].values > thr[METHOD]
    idx = np.where(mask_m)[0]

    positions_df = pd.DataFrame({"spot": spots[idx], "x_img": xs[idx], "y_img": ys[idx], "region_id": spot_region_ids[idx], "laure_region_id": spot_laure_ids[idx]})

    annotations_df = annotate_spot_types(ad, positions_df["spot"].tolist())

    final_df = positions_df.merge(annotations_df, on="spot", how="left")
    final_df.to_csv(f"results/intermediate/pseudobulk/{sample_id}_{METHOD}_complete_metadata.csv", index=False)

pseudobulk_records = []

# Process samples for pseudobulking
for sample_id in samples.keys():
    print(f"Processing pseudobulk for sample {sample_id}")

    metadata = pd.read_csv(f"results/intermediate/pseudobulk/{sample_id}_{METHOD}_complete_metadata.csv")
    ad = sc.read_h5ad(INPUT_DIR / f"{sample_id}_CARDobj_step1_k12.h5ad")

    # Iterate over unique laure_region_ids
    for laure_region_id in metadata['laure_region_id'].unique():
        if laure_region_id == "none":
            continue

        spots_in_region = metadata.query("laure_region_id == @laure_region_id and beta == True")['spot']
        expr_data = ad[spots_in_region].X

        if hasattr(expr_data, "toarray"):
            expr_data = expr_data.toarray()

        # Sum expression per gene for pseudobulk
        expr_sum = np.nansum(expr_data, axis=0)
        genes = ad.var_names

        pseudobulk_records.append({
            "sample_id": sample_id,
            "laure_region_id": laure_region_id,
            **dict(zip(genes, expr_sum))
        })

pseudobulk_df = pd.DataFrame(pseudobulk_records)
pseudobulk_df.to_csv(f"results/intermediate/pseudobulk_merged/{METHOD}_beta_pseudobulk_expression_count_table.csv", index=False)
pseudobulk_df.replace(0,np.nan).dropna(axis=1,thresh=52).replace(np.nan,0).to_csv(f'results/intermediate/pseudobulk_merged/{METHOD}_beta_JC_pseudobulk_count_df.csv')
metadata_df = pd.concat([pd.read_csv(f"results/intermediate/pseudobulk/{sample_id}_{METHOD}_complete_metadata.csv") for sample_id in samples.keys()],axis=0)
metadata_df.to_csv(f'results/intermediate/pseudobulk_merged/{METHOD}_metadata_JC_df.csv')

In [ ]:
metadata_df

## Visualizations (after pseudobulks, diagnostics)

In [ ]:
METHOD = "q75"

group_map = {
    **{s: "control_17W" for s in ["FFPE315", "FFPE319", "FFPE320"]},
    **{s: "aCD3_17W" for s in ["FFPE317", "FFPE318", "FFPE322"]},
    **{s: "E2GLP1_17W" for s in ["FFPE801", "FFPE802", "FFPE803"]},
    **{s: "combo_17W" for s in ["FFPE316", "FFPE323"]},
    **{s: "control_12W" for s in ["Mouse1", "Mouse2", "W12F"]},
}

all_metadata = []

for sample_id, group_label in group_map.items():
    metadata = pd.read_csv(f"results/intermediate/pseudobulk/{sample_id}_{METHOD}_complete_metadata.csv")
    metadata['sample_id'] = sample_id
    metadata['group'] = group_label
    all_metadata.append(metadata)

df_metadata = pd.concat(all_metadata, ignore_index=True)

# Calculate metrics
region_summary = df_metadata.groupby(['sample_id', 'group', 'laure_region_id']).agg(
    total_spots=('spot', 'count'),
    immune_spots=('panleuko', 'sum')
).reset_index()

region_summary['immune_fraction'] = region_summary['immune_spots'] / region_summary['total_spots']
region_summary['has_immune'] = region_summary['immune_spots'] > 0

# Number of regions with/without immune (aggregated by group)
region_counts = region_summary.groupby(['group']).agg(
    total_regions=('laure_region_id', 'count'),
    immune_regions=('has_immune', 'sum')
).reset_index()

# Mean immune fraction per region with any immune spots (aggregated by group)
immune_fraction_summary = region_summary[region_summary['has_immune']].groupby(['group']).agg(
    mean_immune_fraction=('immune_fraction', 'mean')
).reset_index()

# Prepare data explicitly for stacked barplot
region_counts['without_immune'] = region_counts['total_regions'] - region_counts['immune_regions']
groups = region_counts['group'].values
immune_counts = region_counts['immune_regions'].values
without_immune_counts = region_counts['without_immune'].values

# Plot stacked bars clearly
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].bar(groups, immune_counts, color='#FF7F7F', edgecolor='black', label='With immune')
axes[0].bar(groups, without_immune_counts, bottom=immune_counts, color='lightgrey', edgecolor='black', label='Without immune')

axes[0].set_ylabel('Number of regions')
axes[0].set_xticklabels(groups, rotation=90)
# axes[0].legend()

# Mean immune-spot fraction plot
sns.barplot(
    data=immune_fraction_summary,
    x='group', y='mean_immune_fraction', color='#FF7F7F', ax=axes[1], edgecolor='black'
)
axes[1].set_ylabel('Mean immune fraction\n(per immune region)')
axes[1].set_xlabel('')
axes[1].set_ylim(0,1)
axes[1].tick_params(axis='x', rotation=90)

sns.despine(trim=True)
plt.tight_layout()
plt.show()

# Define the desired order
group_order = ["control_12W", "control_17W", "E2GLP1_17W", "aCD3_17W", "combo_17W"]

# Calculate region counts per sample and group
region_counts_sample = region_summary.groupby(['group', 'sample_id']).agg(
    total_regions=('laure_region_id', 'count'),
    immune_regions=('has_immune', 'sum')
).reset_index()

region_counts_sample['without_immune'] = region_counts_sample['total_regions'] - region_counts_sample['immune_regions']

# Proper aggregation (mean and standard deviation across samples within groups)
region_counts = region_counts_sample.groupby('group').agg(
    mean_total_regions=('total_regions', 'mean'),
    se_total_regions=('total_regions', lambda x: np.std(x, ddof=1)/np.sqrt(len(x))),
    mean_immune_regions=('immune_regions', 'mean'),
    se_immune_regions=('immune_regions', lambda x: np.std(x, ddof=1)/np.sqrt(len(x))),
    mean_without_immune=('without_immune', 'mean'),
    se_without_immune=('without_immune', lambda x: np.std(x, ddof=1)/np.sqrt(len(x)))
).reset_index()

# Mean immune fraction per region with any immune spots (aggregated correctly by group)
immune_fraction_summary = (
    region_summary[region_summary['has_immune']]
    .groupby(['group', 'sample_id'])
    .agg(mean_immune_fraction=('immune_fraction', 'mean'))
    .reset_index()
    .groupby('group')
    .agg(
        mean_immune_fraction=('mean_immune_fraction', 'mean'),
        std_immune_fraction=('mean_immune_fraction', 'std')
    ).reindex(group_order).reset_index()
)

region_counts = pd.concat([region_counts[region_counts['group']==ord_] for ord_ in group_order],axis=0)

# Plot stacked bars with proper error bars and correct ordering
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].bar(
    region_counts['group'], region_counts['mean_immune_regions'],
    yerr=region_counts['se_immune_regions'], 
    color='darkgray', edgecolor='black', label='With immune', capsize=5
)

axes[0].bar(
    region_counts['group'], region_counts['mean_without_immune'],
    bottom=region_counts['mean_immune_regions'],
    yerr=region_counts['se_without_immune'], 
    color='lightgray', edgecolor='black', label='Without immune', capsize=5
)

axes[0].set_ylabel('Number of islets (type)')
axes[0].tick_params(axis='x', rotation=90)
axes[0].legend()

# Mean immune-spot fraction plot with error bars and correct order
sns.barplot(
    data=immune_fraction_summary,
    x='group', y='mean_immune_fraction', color='darkgray',
    ax=axes[1], edgecolor='black',
    capsize=0.2, yerr=immune_fraction_summary['std_immune_fraction']
)

axes[1].set_ylabel('Immune spot fraction\n(per immune inf. islets)')
axes[1].set_xlabel('')
axes[1].set_ylim(0,1)
axes[1].tick_params(axis='x', rotation=90)

sns.despine(trim=True)
plt.tight_layout()
plt.show()


df = pd.read_csv(f'results/intermediate/pseudobulk_merged/{METHOD}_metadata_JC_df.csv')
group_map = {
    **{s: "untreated_17w" for s in ["FFPE315", "FFPE319", "FFPE320"]},
    **{s: "mono_aCD3_17w" for s in ["FFPE317", "FFPE318", "FFPE322"]},
    **{s: "mono_E2GLP1_17w" for s in ["FFPE801", "FFPE802", "FFPE803"]},
    **{s: "combo_aCD3_E2GLP1_17w" for s in ["FFPE316", "FFPE323"]},
    **{s: "untreated_12w" for s in ["Mouse1", "Mouse2", "W12F"]},
}

df['sample'] = [l.split('_')[0] if 'Laure' in l else np.nan for l in df['laure_region_id']]
df.dropna(subset='sample',inplace=True)


df['group'] = df['sample'].map(group_map)

group_order = [
    "untreated_12w", "untreated_17w", "mono_aCD3_17w",
    "mono_E2GLP1_17w", "combo_aCD3_E2GLP1_17w"
]
celltype_order = ["panleuko", "Tcell", "CD8pos"]

results = []
for (group, sample), group_df in df.groupby(['group', 'sample']):
    total_spots = len(group_df)
    for celltype in celltype_order:
        pos = group_df[celltype].sum()
        frac = pos / total_spots if total_spots else 0
        results.append({'group': group, 'sample': sample, 'celltype': celltype, 'fraction': frac})

frac_df = pd.DataFrame(results)
summary = frac_df.groupby(['group', 'celltype'])['fraction'].agg(['mean', 'sem']).reset_index()
means = summary.pivot(index='group', columns='celltype', values='mean').loc[group_order, celltype_order]
sems = summary.pivot(index='group', columns='celltype', values='sem').loc[group_order, celltype_order]

colors = {
    "panleuko": "#77bfa3",
    "Tcell": "#f6a06b",
    "CD8pos": "#838ac9"
}

fig, ax = plt.subplots(figsize=(5,3))
bar_width = 0.25
x = np.arange(len(group_order))

for i, celltype in enumerate(celltype_order):
    ax.bar(
        x + i * bar_width,
        means[celltype],
        width=bar_width,
        color=colors[celltype],
        label=celltype,
        linewidth=0,
        zorder=2
    )
    # Error bars with no capsize (no whiskers)
    ax.errorbar(
        x + i * bar_width, means[celltype], yerr=sems[celltype],
        fmt='none', ecolor='black', elinewidth=1.5, capsize=0, zorder=3
    )

ax.set_xticks(x + bar_width)
ax.set_xticklabels(group_order, rotation=30, ha='right')
ax.set_ylabel('Spot Fraction (per islet)')
ax.legend(frameon=True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle=':', linewidth=0.7, alpha=0.5)
plt.tight_layout()
plt.show()